# E2.2 · Horizontal AI regulation

**Function E — AI Governance for Agentic Systems → Building the Governance Platform — Regulatory and Compliance**  ·  *Security of AI*

Builds on **[E2.1 · The regulatory map](https://spbreed.github.io/cyber-commons/lessons/E2.1.html)**.

| | |
|---|---|
| Tools used | standard library only |

## What this lesson is

**What it covers.** Classify three workflows and defend the boundary cases.

**Why a security engineer needs it.** "We only deployed it, we didn't build it" — sometimes true, often not. The control it builds is: risk classification, GPAI obligations, transparency duties, and how agentic deployment changes classification.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Horizontal AI regulation applies to you regardless of sector, and its obligations are structural: risk management, documentation, oversight. Those are programme requirements, not paperwork requirements.

> **At CyberTravels.** Horizontal AI obligations land on CyberTravels as programme requirements: a risk register, technical documentation, named human oversight, measured accuracy, post-market monitoring.

## 2 · The framework

```
   horizontal obligations are structural, not clerical

   risk management system     -> you need a register and a tiering model
   technical documentation    -> generated, not written once
   human oversight            -> a named person with real authority
   accuracy / robustness      -> measured, with the method recorded
   post-market monitoring     -> drift detection, by another name
```

Horizontal AI regulation is, in practice, mostly about **documented process and
human oversight**. That is good news, because those map onto controls you can
build and evidence mechanically.

The trap is answering a clause with a policy document. "We maintain appropriate
human oversight" satisfies nobody who asks the follow-up question, and the
follow-up question is always the same: *show me*.

So the working method is to resolve each regulatory theme down to a control from
your own catalogue (E1.4), and let the control's evidence be the answer. Four
themes cover most of it:

- risk management system,
- record-keeping,
- human oversight,
- accuracy and robustness.

Each one resolves to controls you already built in tracks A, B and D.

## 3 · The procedure, as a skill

The skill maps four regulatory themes to named controls with concrete artefacts, then applies the show-me test to the prose answers a policy currently offers — and counts how many sentences survive it.

In [ ]:
# skills/regulatory/horizontal-requirement-to-control/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: horizontal-requirement-to-control
description: >-
  Turn horizontal AI-regulation themes into named controls with concrete
  evidence artefacts, and apply the show-me test to the prose answers a policy
  currently offers. Use when a regulation is being answered with paragraphs.
allowed-tools: Read, Grep, Glob
---

# Every prose answer fails the show-me test

Horizontal AI regulation resolves to a small number of recurring themes — risk
management, data governance, human oversight, record keeping, transparency,
accuracy and robustness. Each maps to controls you can operate. A policy that
answers them in prose passes reading and fails the first follow-up, which is
always "show me".

## When to use this

Responding to a horizontal AI regulation, and auditing an existing response
before somebody external does.

## Procedure

**1 — Extract the themes rather than the article numbers.** Article numbers
change between drafts and jurisdictions; the themes are stable and map to
controls you already have.

**2 — Map each theme to named controls from your catalogue.** If a theme has no
control, that is the finding — record it as a gap rather than writing a
paragraph.

**3 — Attach the evidence artefact per control.** The specific thing that would
be handed over: a log export, a test result, an approval record, a signed
attestation.

**4 — Apply the show-me test to the current prose answers.** For each sentence,
is there an artefact behind it? Count the sentences that survive. It is usually
none, and the count is more persuasive than the argument.

**5 — Check freshness.** An artefact older than the control's freshness window
evidences the past. Report themes as fully evidenced, stale, or unevidenced —
three states, not two.

## Output contract

```json
{
  "themes": [{"theme": "str", "controls": ["str"], "gap": false}],
  "evidence": [{"control": "str", "artefact": "str", "as_of": "str", "fresh": true}],
  "prose_answers": [{"text": "str", "survives_show_me": false}],
  "status": [{"theme": "str", "state": "evidenced|stale|unevidenced"}]
}
```

## Failure modes

- **Mapping article numbers.** They move; the themes do not.
- **Answering a gap with a paragraph.** It reads as coverage and is not.
- **Two-state reporting.** Stale is the state most of your evidence is in.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/regulatory/horizontal-requirement-to-control/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/regulatory/horizontal-requirement-to-control/scripts/horizontal_requirement_to_control.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Turn horizontal AI regulation themes into named controls with evidence artefacts, and apply the show-me test to prose answers.

This is the executable half of the `horizontal-requirement-to-control` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

CATALOGUE = {
 "AC-1": ("agent identities distinct from human and separately revocable",
          "gateway logs with an act chain; monthly sample"),
 "AC-2": ("delegated authority narrows at every hop",
          "regression suite IDN-01/IDN-04 on every release"),
 "SB-1": ("egress deny-by-default with an allowlist", "90-day denial log"),
 "SB-2": ("privileged tools require approval below L3", "tool policy in git + denial log"),
 "EV-1": ("every action logged with the acting identity", "audit sample of 50 actions"),
 "EV-2": ("accuracy evaluated against a held-out key per release",
          "expert accuracy report with sample size"),
 "DR-1": ("behavioural drift raises an alert", "drift alerts and dispositions"),
 "ST-1": ("a tested stop mechanism you own", "game-day record with measured time-to-stop"),
}
THEMES = {
 "risk management system":       ["AC-1", "SB-2", "DR-1"],
 "record-keeping (Art.12)":      ["EV-1", "AC-2"],
 "human oversight (Art.14)":     ["SB-2", "ST-1"],
 "accuracy and robustness":      ["EV-2", "DR-1"],
}
for theme, cids in THEMES.items():
    print(f"{theme}")
    for cid in cids:
        text, evidence = CATALOGUE[cid]
        print(f"   {cid}  {text}")
        print(f"         evidence: {evidence}")
    print()

WEAK = {
 "risk management system":   "We operate a risk management framework for AI systems.",
 "record-keeping (Art.12)":  "Appropriate logs are retained.",
 "human oversight (Art.14)": "Human oversight is maintained at all times.",
 "accuracy and robustness":  "Models are tested prior to deployment.",
}
def survives_followup(answer, controls):
    """The follow-up question is always 'show me'."""
    return bool(controls), ("names a control with an artefact" if controls
                            else "no artefact — the answer IS the evidence, which is the problem")

print(f"{'theme':28s}{'prose answer survives?':>24}")
print("-" * 56)
for theme in THEMES:
    ok_weak, _ = survives_followup(WEAK[theme], [])
    print(f"{theme:28s}{str(ok_weak):>24}")
print("\nAll four fail the same way: there is nothing to produce when asked.")

import time
from dataclasses import dataclass
now = time.time(); DAY = 86400

@dataclass
class ControlTest:
    cid: str; passed: bool; tested_at: float; valid_for_days: float
    def state(self, at):
        if (at - self.tested_at)/DAY > self.valid_for_days: return "STALE"
        return "PASS" if self.passed else "FAIL"

TESTS = [ControlTest("AC-1", True,  now -  3*DAY, 30),
         ControlTest("AC-2", True,  now -  9*DAY, 30),
         ControlTest("SB-2", True,  now - 40*DAY, 30),
         ControlTest("EV-1", True,  now -  5*DAY, 60),
         ControlTest("EV-2", True,  now - 12*DAY, 30),
         ControlTest("DR-1", False, now,          30),
         ControlTest("ST-1", True,  now - 41*DAY, 180)]
by = {t.cid: t for t in TESTS}

print(f"{'theme':28s}{'controls':22s}{'evidenced now':>15}")
print("-" * 68)
for theme, cids in THEMES.items():
    states = [by[c].state(now) if c in by else "NO EVIDENCE" for c in cids]
    ok = all(s == "PASS" for s in states)
    blockers = ",".join(c for c, s in zip(cids, states) if s != "PASS")
    verdict = "yes" if ok else f"NO — {blockers}"
    print(f"{theme:28s}{str(cids):22s}{verdict:>15}")

fully = [t for t, cids in THEMES.items()
         if all((by[c].state(now) if c in by else "X") == "PASS" for c in cids)]
print(f"\nthemes fully evidenced right now: {len(fully)}/{len(THEMES)}  {fully}")
print("\nThat sentence is what you say to a supervisor. It is smaller than the")
print("prose version and it is defensible, which is the trade worth making.")
assert len(fully) < len(THEMES)

## What you just proved

Four regulatory themes resolve to named controls, each with a concrete evidence artefact. All four prose answers fail the show-me test. Checking freshness, two themes are fully evidenced — human oversight fails on a stale SB-2 and risk management on a failing DR-1 — giving a smaller but defensible statement.

## Your turn

Take one clause your programme claims to satisfy and trace it to an artefact with a date. If the trail ends at a policy document, the clause is ticked and undefended.

---

**Next → [E2.3 · Voluntary frameworks as your spine](https://spbreed.github.io/cyber-commons/lessons/E2.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E2.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E2.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*